In [13]:
import duckdb
import pandas as pd
import os
from datetime import datetime


In [14]:
#Conexão com o Banco
con = duckdb.connect(database='dados_duckdb.db' , read_only=False)

In [22]:
#CRIAR DATAFRAME e dizer que o separador é ';'
arquivo = 'z0019_2.csv' #ler o arquivo
data_ingestao = datetime.now() #cria a hora que foi inserida na tabela
df = pd.read_csv(f'../landing/{arquivo}' , sep=';')
df['nome_arquivo'] = arquivo #cria a coluna nome do arquivo e coloca o nome do arquivo nela
df['data_ingestao'] = data_ingestao #cria a coluna data de ingestão
df.head() #LER O DATAFAME

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-04-19 13:27:11.905987
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-04-19 13:27:11.905987
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-04-19 13:27:11.905987


In [23]:
#ingestão desses arquivos em uma tabla bronze
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos (
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
        )   
""")

In [30]:
#CONSULTA SIMPLES NA TABELA CRIADA - VAI RETORNAR VAZIO
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(6)    

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-04-19 13:20:37.768055
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-04-19 13:20:37.768055
2,10003,PREGO,BT10,100,50,z0019_1.csv,2026-04-19 13:20:37.768055
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-04-19 13:27:11.905987
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-04-19 13:27:11.905987
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-04-19 13:27:11.905987


In [ ]:
#INGESTÃO DOS DADOS NA TABELA CRIADA - CAMADA BRONZE
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [ ]:
#RENOMEANDO A TABELA PARA O NOME BRUTO DO ARQUIVO
con.execute ("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

In [31]:
#FINALIZAR A CONEXÃO
con.close()